# 4.8. Cluster Sample Viewer

Plots a stratified random sample of labeled clusters across all `SETUP_TILECODES` tiles.
Each cluster is shown as a **top view (XY)** and a **side view (XZ)**, colored by height above ground.
Label name and origin tilecode are shown per cluster.

## Config

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from config import CLUSTERS_DIR

N_PER_CLASS = 4   # samples to show per label class
SEED        = 42
SKIP_LABELS = {0, 99}  # unknown and noise

## Load inventory

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

from utils.labels import Labels
from utils.cluster_io import load_cluster_npz

inv_path = CLUSTERS_DIR / 'inventory.csv'
if inv_path.exists():
    df = pd.read_csv(inv_path)
else:
    parts = list(CLUSTERS_DIR.glob('inventory_*.csv'))
    if not parts:
        raise FileNotFoundError(f'No inventory CSV found in {CLUSTERS_DIR}')
    df = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)

# Use final_label when reviewed, else fall back to auto label
df['effective_label'] = (
    df['final_label'].where(df['final_label'].notna(), df['label']).astype(int)
)
df = df[~df['effective_label'].isin(SKIP_LABELS)].copy()

print(f'Labeled clusters: {len(df)}')
dist = df['effective_label'].map(lambda l: Labels.STR_DICT.get(l, str(l))).value_counts()
print(dist.to_string())

## Stratified sample

In [ ]:
sample = pd.concat([
    g.sample(n=min(N_PER_CLASS, len(g)), random_state=SEED)
    for _, g in df.groupby('effective_label')
]).reset_index(drop=True)
classes = sorted(sample['effective_label'].unique())
print(f'{len(sample)} clusters sampled across {len(classes)} classes')

## Plot: top view + side view per class

In [ ]:
def resolve_npz(row):
    """Return the NPZ path, remapping if the stored path doesn't exist on this device."""
    p = Path(row['npz_path'])
    if p.exists():
        return p
    # Fall back: reconstruct from CLUSTERS_DIR + tilecode + filename
    alt = CLUSTERS_DIR / row['tilecode'] / p.name
    return alt if alt.exists() else None


def plot_view(ax, xs, ys, hag, title, xlabel, ylabel):
    norm   = Normalize(vmin=0, vmax=max(float(hag.max()), 0.1))
    colors = cm.viridis(norm(hag))
    s      = max(0.5, 3000 / len(xs))
    ax.scatter(xs, ys, c=colors, s=s, linewidths=0, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=7, pad=2)
    ax.set_xlabel(xlabel, fontsize=6)
    ax.set_ylabel(ylabel, fontsize=6)
    ax.tick_params(labelsize=5)


for lbl in classes:
    label_name  = Labels.STR_DICT.get(lbl, str(lbl))
    class_rows  = sample[sample['effective_label'] == lbl].reset_index(drop=True)
    n_samples   = len(class_rows)

    fig, axes = plt.subplots(
        2, n_samples,
        figsize=(n_samples * 3, 6),
        gridspec_kw={'hspace': 0.4, 'wspace': 0.3},
    )
    if n_samples == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(f'{label_name}  (label {lbl})', fontsize=11, y=1.01, fontweight='bold')

    for col_i, row in class_rows.iterrows():
        npz_path = resolve_npz(row)
        ax_top  = axes[0, col_i]
        ax_side = axes[1, col_i]

        if npz_path is None:
            for ax in (ax_top, ax_side):
                ax.text(0.5, 0.5, 'file not found', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='red')
                ax.axis('off')
            continue

        try:
            data = load_cluster_npz(npz_path)
        except Exception as e:
            for ax in (ax_top, ax_side):
                ax.text(0.5, 0.5, str(e)[:40], ha='center', va='center',
                        transform=ax.transAxes, fontsize=7, color='red')
                ax.axis('off')
            continue

        xyz = data['xyz_centered']  # (N,3) — XY centred, Z absolute
        hag = data['height_ag']     # (N,)  height above ground

        n_pts = data.get('n_raw_pts', len(xyz))
        area  = data.get('area_m2', 0.0)
        if hasattr(n_pts, 'item'): n_pts = n_pts.item()
        if hasattr(area,  'item'): area  = area.item()

        subtitle = f"{row['tilecode']}\n{int(n_pts)} pts  {area:.1f} m²"

        # Top view: XY, colored by height above ground
        plot_view(ax_top,  xyz[:, 0], xyz[:, 1], hag, subtitle, 'X (m)', 'Y (m)')

        # Side view: along the longest horizontal extent
        if np.ptp(xyz[:, 0]) >= np.ptp(xyz[:, 1]):
            plot_view(ax_side, xyz[:, 0], hag, hag, '', 'X (m)', 'height (m)')
        else:
            plot_view(ax_side, xyz[:, 1], hag, hag, '', 'Y (m)', 'height (m)')

    # Row labels
    axes[0, 0].annotate('top view',  xy=(-0.35, 0.5), xycoords='axes fraction',
                         rotation=90, va='center', fontsize=8)
    axes[1, 0].annotate('side view', xy=(-0.35, 0.5), xycoords='axes fraction',
                         rotation=90, va='center', fontsize=8)

    plt.show()